# Milestone 4 - Task 7: Export Model-Ready Datasets

**Owner:** Sanjeewa Narayana  
**Objective:** Take the two feature matrices (email track from M4-T6, URL track from M4-T3), validate them for ML, and export **stratified train/test splits** so that every candidate model in M5 is trained and evaluated on the *identical* data — a prerequisite for fair model comparison.

### Two model-ready datasets
| Track | Matrix | Rows | Feature columns |
|---|---|---|---|
| Email | email_features.csv | 82,078 | text_clean (text) + 5 numeric |
| URL   | url_features.csv   | 641,119 | 11 numeric (url is an identifier, not a feature) |

### ML-compatibility guarantees produced here
- **No NaN / no empty required cells** (one empty text_clean row is repaired).
- **Identifier columns excluded from features** (`url` kept for reference only).
- **Stratified 80/20 split** with a fixed seed → identical splits for all models.
- **Label is integer 0/1** and class balance is preserved in both splits.

## Step 0 — Install deps and download inputs
```bash
pip install pandas scikit-learn
mkdir -p data/processed
aws s3 cp s3://email-security-pipeline-datasets/datasets/processed/email_features.csv data/processed/email_features.csv --profile lab-user
aws s3 cp s3://email-security-pipeline-datasets/datasets/processed/url_features.csv   data/processed/url_features.csv   --profile lab-user
```

In [2]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROC = ROOT / 'data' / 'processed'
RANDOM_STATE = 42   # fixed seed -> every model uses the same split
TEST_SIZE = 0.20

## Step 1 — Load and quality-check both matrices

In [3]:
email = pd.read_csv(PROC / 'email_features.csv')
url   = pd.read_csv(PROC / 'url_features.csv')

print('EMAIL:', email.shape, '| NaN:', int(email.isnull().sum().sum()))
print('  label dist:', email['label'].value_counts().to_dict())
print('URL  :', url.shape, '| NaN:', int(url.isnull().sum().sum()))
print('  label dist:', url['label'].value_counts().to_dict())

EMAIL: (82078, 7) | NaN: 1
  label dist: {1: 42845, 0: 39233}
URL  : (641119, 13) | NaN: 0
  label dist: {0: 428080, 1: 213039}


## Step 2 — Repair for ML compatibility
Normalization can leave an email with empty text (e.g. a body that was entirely HTML/URLs). That produces a NaN in `text_clean`, which crashes text vectorizers. We replace any missing text with an empty string and confirm zero NaN remain.

In [4]:
# Fix empty/NaN text so TF-IDF never sees a NaN
email['text_clean'] = email['text_clean'].fillna('').astype(str)
email['label'] = email['label'].astype(int)
url['label'] = url['label'].astype(int)

assert email.isnull().sum().sum() == 0, 'email still has NaN'
assert url.drop(columns=['url']).isnull().sum().sum() == 0, 'url features have NaN'
print('Repaired. email NaN:', int(email.isnull().sum().sum()),
      '| url feature NaN:', int(url.drop(columns=["url"]).isnull().sum().sum()))

Repaired. email NaN: 0 | url feature NaN: 0


## Step 3 — Define feature groups (what the model should use)
Explicit column groups so M5 modeling code is unambiguous.

In [5]:
# EMAIL track
EMAIL_TEXT_COL    = 'text_clean'                # -> TF-IDF / n-grams
EMAIL_NUMERIC_COLS = ['urgency_score','link_count','html_ratio',
                      'word_count','avg_word_length']
EMAIL_TARGET = 'label'

# URL track ('url' is an identifier, NOT a feature)
URL_ID_COL = 'url'
URL_FEATURE_COLS = ['url_length','hostname_length','num_dots','num_hyphens',
                    'num_at','num_digits','num_special_chars','has_ip',
                    'has_https','num_subdomains','is_shortened']
URL_TARGET = 'label'

print('email features:', [EMAIL_TEXT_COL] + EMAIL_NUMERIC_COLS)
print('url features  :', URL_FEATURE_COLS)

email features: ['text_clean', 'urgency_score', 'link_count', 'html_ratio', 'word_count', 'avg_word_length']
url features  : ['url_length', 'hostname_length', 'num_dots', 'num_hyphens', 'num_at', 'num_digits', 'num_special_chars', 'has_ip', 'has_https', 'num_subdomains', 'is_shortened']


## Step 4 — Stratified train/test split
Stratifying on the label keeps the phishing/legit ratio identical in train and test. The fixed seed means anyone re-running M5 gets the exact same rows.

In [6]:
e_train, e_test = train_test_split(
    email, test_size=TEST_SIZE, stratify=email[EMAIL_TARGET], random_state=RANDOM_STATE)
u_train, u_test = train_test_split(
    url, test_size=TEST_SIZE, stratify=url[URL_TARGET], random_state=RANDOM_STATE)

print(f'EMAIL  train={len(e_train):>6}  test={len(e_test):>6}')
print('   train balance:', e_train[EMAIL_TARGET].value_counts(normalize=True).round(3).to_dict())
print('   test  balance:', e_test[EMAIL_TARGET].value_counts(normalize=True).round(3).to_dict())
print(f'URL    train={len(u_train):>6}  test={len(u_test):>6}')
print('   train balance:', u_train[URL_TARGET].value_counts(normalize=True).round(3).to_dict())

EMAIL  train= 65662  test= 16416
   train balance: {1: 0.522, 0: 0.478}
   test  balance: {1: 0.522, 0: 0.478}
URL    train=512895  test=128224
   train balance: {0: 0.668, 1: 0.332}


## Step 5 — Save the model-ready splits

In [7]:
e_train.to_csv(PROC / 'email_train.csv', index=False)
e_test.to_csv(PROC / 'email_test.csv', index=False)
u_train.to_csv(PROC / 'url_train.csv', index=False)
u_test.to_csv(PROC / 'url_test.csv', index=False)
for name in ['email_train','email_test','url_train','url_test']:
    p = PROC / f'{name}.csv'
    print(f'{name:12s} {p.stat().st_size/1e6:7.1f} MB')

email_train     80.4 MB
email_test      19.1 MB
url_train       44.7 MB
url_test        11.1 MB


## Step 6 — Upload splits to S3 (run in terminal; data stays out of Git)
```bash
for f in email_train email_test url_train url_test; do
  aws s3 cp data/processed/$f.csv \
    s3://email-security-pipeline-datasets/datasets/processed/splits/$f.csv \
    --profile lab-user
done
```

## Step 7 — Modeling guidance (M5): how to load for multiple models
All candidate models share the saved splits. Use a `ColumnTransformer` so the same feature pipeline feeds every estimator. **Model-specific note:** scale numeric features for distance/linear models (LogisticRegression, SVM); tree ensembles (RandomForest, GradientBoosting/XGBoost) need no scaling; MultinomialNB needs non-negative input, so use TF-IDF only or MinMax (not StandardScaler) for it.

```python
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, roc_auc_score

tr = pd.read_csv('data/processed/email_train.csv')
te = pd.read_csv('data/processed/email_test.csv')
tr['text_clean'] = tr['text_clean'].fillna('')
te['text_clean'] = te['text_clean'].fillna('')

NUM = ['urgency_score','link_count','html_ratio','word_count','avg_word_length']
pre = ColumnTransformer([
    ('text', TfidfVectorizer(max_features=20000, ngram_range=(1,2)), 'text_clean'),
    ('num',  StandardScaler(with_mean=False), NUM),
])

models = {
    'logreg': LogisticRegression(max_iter=1000),
    'svm':    LinearSVC(),
    'rf':     RandomForestClassifier(n_estimators=200, n_jobs=-1),
}
for name, clf in models.items():
    pipe = Pipeline([('pre', pre), ('clf', clf)]).fit(tr, tr['label'])
    pred = pipe.predict(te)
    print(name); print(classification_report(te['label'], pred))
```

## Conclusion
M4-T7 exports two validated, model-ready datasets — an email feature set (text + 5 numeric) and a URL feature set (11 numeric) — each split into stratified train/test partitions with a fixed seed. NaN/empty cells are repaired and identifier columns are excluded, so the data drops directly into scikit-learn pipelines. Because all models share identical splits, M5 can compare multiple classifiers fairly to select the best fit for each track.